# Q34 — untouched cross-archive boundary-child replication

## tl;dr

Q34 transferred Q33B's frozen ARA route from a random network ordering to the
previously untouched public `pure_greedy` archive.

The exact route retained a positive median in both branches, but its positive
fraction fell from 63.64% to 54.21% and it did not reliably beat all
same-rule controls.

Frozen verdict: **CROSS-ARCHIVE BOUNDARY-CHILD FLOW NOT REPLICATED**

Independent raw-HDF5 validation: **PASS**.


## Context & Methods

The invariant geometry remained:

\[
2+\left(1+\frac12\right)=3.5.
\]

`0.5` is a declared adjacent-rung projection, not a coefficient fitted to the
target. Starting normalized closure selects the lower endpoint child; its
unseen next-slice closure change is scored.

Development scales, source conditions, event sample, controls and every
pass/fail threshold are identical to Q33B and were sealed before download.


In [1]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / "Q34_CROSS_ARCHIVE_BOUNDARY_CHILD_RESULTS.json").exists():
    ROOT = Path(r"F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\quantum")

result = json.loads((ROOT / "Q34_CROSS_ARCHIVE_BOUNDARY_CHILD_RESULTS.json").read_text())
validation = json.loads((ROOT / "Q34_CROSS_ARCHIVE_BOUNDARY_CHILD_VALIDATION.json").read_text())
evaluation = result["splits"]["evaluation"]
print("Structural path:", result["geometry"]["complete_path"])
print("Evaluation events:", evaluation["source_events"])
print("Verdict:", result["frozen_verdict"]["label"])
print("Independent validation:", validation["status"])


Structural path: 3.5
Evaluation events: 16001
Verdict: CROSS-ARCHIVE BOUNDARY-CHILD FLOW NOT REPLICATED
Independent validation: PASS


## Results

### Route flow


In [2]:
routes = pd.DataFrame([
    {
        "route": route,
        "events": evaluation["routes"][route]["paired_events"],
        "median flow": evaluation["routes"][route]["flow"]["median"],
        "mean flow": evaluation["routes"][route]["flow"]["mean"],
        "positive fraction": evaluation["routes"][route]["positive_fraction"],
        "median starting z": evaluation["routes"][route]["start_z"]["median"],
    }
    for route in ("exact", "sibling", "topology", "seed", "time")
])
routes


route,events,median flow,mean flow,positive fraction,median starting z
exact,16001,0.007357,0.035773,0.542091,0.172048
sibling,16001,0.014607,0.021483,0.519343,0.874076
topology,16001,0.002027,0.030082,0.534842,0.108502
seed,15085,0.003401,0.043820,0.572423,0.076566
time,15667,0.010884,0.057345,0.600881,0.144865


The exact route remains mildly positive. It exceeds the sibling
by 2.27 percentage points in positive frequency, but the sibling has a larger
marginal median and the seed/time controls are more often positive.


### Paired comparisons


In [3]:
paired = pd.DataFrame([
    {
        "comparator": comparator,
        "paired median difference": evaluation["paired_differences"][comparator]["median"],
        "cluster mean difference": result["evaluation_bootstrap"][comparator]["mean_exact_minus_comparator"],
        "bootstrap P(exact greater)": result["evaluation_bootstrap"][comparator]["probability_exact_greater"],
    }
    for comparator in ("sibling", "topology", "seed", "time")
])
paired


comparator,paired median difference,cluster mean difference,bootstrap P(exact greater)
sibling,-0.011223,0.011030,0.7500
topology,0.001621,0.031006,1.0000
seed,-0.004653,0.012410,0.9900
time,-0.003160,0.004914,0.8185


### Branches


In [4]:
branches = pd.DataFrame([
    {
        "branch": branch,
        "events": evaluation["branches"][branch]["source_events"],
        "median exact flow": evaluation["branches"][branch]["exact_flow"]["median"],
        "positive fraction": evaluation["branches"][branch]["exact_positive_fraction"],
    }
    for branch in ("c2", "c4")
])
branches


branch,events,median exact flow,positive fraction
c2,11584,0.010951,0.531509
c4,4417,0.004851,0.569844


### Cross-archive attenuation


In [5]:
comparison = pd.DataFrame([
    {
        "quantity": "exact median flow",
        "Q33B random": result["q33b_comparison"]["q33b_median_flow"],
        "Q34 greedy": result["q33b_comparison"]["q34_median_flow"],
        "change": result["q33b_comparison"]["median_flow_delta"],
    },
    {
        "quantity": "exact positive fraction",
        "Q33B random": result["q33b_comparison"]["q33b_positive_fraction"],
        "Q34 greedy": result["q33b_comparison"]["q34_positive_fraction"],
        "change": result["q33b_comparison"]["positive_fraction_delta"],
    },
])
comparison


quantity,Q33B random,Q34 greedy,change
exact median flow,0.041425,0.007357,-0.034069
exact positive fraction,0.636403,0.542091,-0.094312


### Frozen-gate result

![Q34 geometry](Q34_CROSS_ARCHIVE_BOUNDARY_CHILD_GEOMETRY.png)

Failed frozen gates:

- `bootstrap_probability_ge_095_vs_sibling`
- `bootstrap_probability_ge_095_vs_time`
- `exact_positive_fraction_ge_055`
- `median_paired_flow_advantage_positive_vs_seed`
- `median_paired_flow_advantage_positive_vs_sibling`
- `median_paired_flow_advantage_positive_vs_time`
- `positive_fraction_advantage_ge_002_vs_seed`
- `positive_fraction_advantage_ge_002_vs_time`
- `positive_fraction_advantage_ge_002_vs_topology`


## Takeaways

1. The untouched target gives a valid negative replication result.
2. A weak inward tendency survives, but Q33B's stronger routing advantage
   does not.
3. The unchanged rule is not invariant to random → greedy network ordering.
4. A network-identity or orientation-aware revision is now a new hypothesis
   requiring a newly frozen test; it cannot rescue Q34 retrospectively.
5. Q34 tests one local ARA route, not universal ARA, physical hardware,
   entanglement, Phase B or the dark sector.

## Data and validation caveats

The target is simulated and exactly diagonal in its sampled connected
correlations. Raw density-matrix reconstruction passed, but the validator
gate-checks saved cluster-bootstrap probabilities rather than redrawing them.
Event-weighted and equal-stratum effects are different estimands.
